# COCO Ground-Truth Data Mining + YOLO11n Benchmark (Full Klygo Mining Suite)

Notebook thực nghiệm môn **Khai phá dữ liệu (Data Mining)** sử dụng **100% High-Level API của Klygo**, chia nhỏ tối đa theo nguyên tắc **mỗi cell chỉ làm 1 việc / in tối đa 1 bảng / vẽ 1 hình**:
- **Làm sạch & phân tích**: `mining.find_duplicates`, `mining.deduplicate`, `mining.stats`, `mining.score`, `mining.cluster`, `mining.compare`
- **11 chiến lược chọn mẫu cùng ngân sách (`TRAIN_BUDGET = 550`)**:
  `random`, `rare_first`, `density`, `weighted`, `quality`, `boxes`, `coverage`, `quota`, `diversity`, `clustering`, `balance`
- **Huấn luyện & Đánh giá**: `models.load("yolo11n.pt")`, `detector.model.val(...)`, `repredict()`, `metrics.evaluate()`
- **Báo cáo & Biểu đồ 300 DPI**: `reports.*`, `utils.tabulate`, `graphics.*`, `archive.compress`

## 1. Cài đặt & Khởi tạo môi trường

In [ ]:
!pip -q install "klygo[ultralytics] @ git+https://github.com/IchigoMazone/klygo.git@extract_hyper_return_type"

In [ ]:
import ssl
try:
    _ssl_ctx = ssl._create_unverified_context()
    ssl._create_default_https_context = lambda: _ssl_ctx
except Exception:
    pass

import platform
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import torch
import klygo
from klygo import (
    archive, config, datasets, files, graphics,
    metrics, mining, models, outputs, postprocessing, reports, utils,
)

# --- Globals & Tham số thực nghiệm (chế độ test nhanh pipeline trước khi chạy full) ---
SEED = 42
MULTI_SEEDS = (42, 100, 2024)       # 3 seeds kiểm chứng cho random + Top-3
BUDGET_GRID = (50, 100, 150)        # test nhanh (full: (250, 550, 1000))
POOL_LIMIT = 400                    # test nhanh (full: 5000)
VAL_LIMIT = 100                     # test nhanh (full: 1000)
TRAIN_BUDGET = 100                  # test nhanh (full: 550)
N_CLUSTERS = 25                     # số cụm cho chiến lược clustering
DEDUP_THRESHOLD = 0.95              # ngưỡng pHash khử trùng lặp

MODEL_NAME = "yolo11n.pt"
EPOCHS = 2                          # test nhanh (full: 50)
IMGSZ = 640
BATCH = 16
WORKERS = 4
DEVICE = 0 if torch.cuda.is_available() else "cpu"
DOWNLOAD_WORKERS = 16
RUN_TRAINING = True
RUN_MULTI_SEED = True               # bật/tắt kiểm chứng 3 seeds (random + Top-3)
RUN_BUDGET_SCALING = True           # bật/tắt khảo sát ngân sách

# Khi chạy FULL báo cáo, mở dòng dưới:
# POOL_LIMIT = 5000; VAL_LIMIT = 1000; TRAIN_BUDGET = 550; BUDGET_GRID = (250, 550, 1000); EPOCHS = 50

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = files.path("/content/coco_mining_project")
RAW_DIR = ROOT / "raw_coco"
ANN_DIR = RAW_DIR / "annotations"
ARTIFACTS = ROOT / "artifacts"
LOG_DIR = ARTIFACTS / "logs"
REPORT_DIR = ARTIFACTS / "reports"
CHECKPOINT_DIR = ARTIFACTS / "checkpoints"
DATASET_DIR = ARTIFACTS / "datasets"
RUNS_DIR = ARTIFACTS / "runs"

for folder in [ANN_DIR, RAW_DIR / "train2017", RAW_DIR / "val2017", LOG_DIR, REPORT_DIR, CHECKPOINT_DIR, DATASET_DIR, RUNS_DIR]:
    files.mkdir(folder)
# Clean up malformed directories if any from interrupted downloads
for p in [RAW_DIR / "annotations_trainval2017.zip", ANN_DIR / "annotations_trainval2017.zip"]:
    if files.is_dir(p):
        files.remove(p, missing_ok=True)


In [ ]:
env_info = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "klygo": getattr(klygo, "__version__", "unknown"),
    "seed": SEED,
}
files.save(LOG_DIR / "01_environment.json", env_info, overwrite=True, verbose=False)
utils.tabulate(env_info, orient="horizontal", title="Bảng 1: Thông số môi trường phần cứng & phần mềm")

In [ ]:
exp_config = {
    "dataset": "COCO 2017",
    "pool_limit": POOL_LIMIT,
    "val_limit": VAL_LIMIT,
    "train_budget": TRAIN_BUDGET,
    "n_clusters": N_CLUSTERS,
    "dedup_threshold": DEDUP_THRESHOLD,
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
}
files.save(LOG_DIR / "02_experiment_config.json", exp_config, overwrite=True, verbose=False)
utils.tabulate(exp_config, title="Bảng 2: Thiết lập tham số thực nghiệm")

## 2. Tải dữ liệu COCO 2017 & Chuyển sang định dạng YOLO

In [ ]:
ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANNOTATIONS_ZIP = RAW_DIR / "annotations_trainval2017.zip"
TRAIN_ANN = RAW_DIR / "annotations" / "instances_train2017.json"
VAL_ANN = RAW_DIR / "annotations" / "instances_val2017.json"

for p in [ANNOTATIONS_ZIP, ANN_DIR / "annotations_trainval2017.zip"]:
    if files.is_dir(p):
        files.remove(p, missing_ok=True)

if not files.is_file(ANNOTATIONS_ZIP):
    files.download(ANNOTATIONS_URL, output_dir=RAW_DIR, overwrite=True, verbose=True, timeout=120.0, retries=3)

if not (files.is_file(TRAIN_ANN) and files.is_file(VAL_ANN)):
    archive.extract(ANNOTATIONS_ZIP, output_dir=RAW_DIR, overwrite=True, verbose=True)

assert files.is_file(TRAIN_ANN) and files.is_file(VAL_ANN)


In [ ]:
rng = np.random.default_rng(SEED)
train_coco = files.load(TRAIN_ANN, verbose=False)
train_all = sorted(train_coco["images"], key=lambda x: x["id"])
pool_images = [train_all[i] for i in np.sort(rng.choice(len(train_all), size=min(POOL_LIMIT, len(train_all)), replace=False))]
pool_ids = {int(im["id"]) for im in pool_images}
pool_anns = [a for a in train_coco["annotations"] if int(a["image_id"]) in pool_ids and not a.get("iscrowd", 0)]

POOL_JSON = RAW_DIR / "pool_train2017.json"
files.save(POOL_JSON, {"images": pool_images, "annotations": pool_anns, "categories": train_coco["categories"]}, overwrite=True, verbose=False)

In [ ]:
val_coco = files.load(VAL_ANN, verbose=False)
val_all = sorted(val_coco["images"], key=lambda x: x["id"])
val_images = [val_all[i] for i in np.sort(rng.choice(len(val_all), size=min(VAL_LIMIT, len(val_all)), replace=False))]
val_ids = {int(im["id"]) for im in val_images}
val_anns = [a for a in val_coco["annotations"] if int(a["image_id"]) in val_ids and not a.get("iscrowd", 0)]

VAL_JSON = RAW_DIR / "val2017_subset.json"
files.save(VAL_JSON, {"images": val_images, "annotations": val_anns, "categories": val_coco["categories"]}, overwrite=True, verbose=False)

In [ ]:
utils.tabulate(
    [
        {"split": "Candidate Pool (train2017)", "images": len(pool_images), "boxes": len(pool_anns), "classes": len(train_coco["categories"])},
        {"split": "Fixed Validation (val2017)", "images": len(val_images), "boxes": len(val_anns), "classes": len(val_coco["categories"])},
    ],
    title="Bảng 3: Tóm tắt mẫu COCO trích xuất",
)

In [ ]:
for im in pool_images:
    p = RAW_DIR / "train2017" / im["file_name"]
    if files.is_dir(p):
        files.remove(p, missing_ok=True)

pool_tasks = [
    (im.get("coco_url") or f"http://images.cocodataset.org/train2017/{im['file_name']}")
    for im in pool_images
    if not files.is_file(RAW_DIR / "train2017" / im["file_name"])
]
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
    futs = [ex.submit(files.download, url, RAW_DIR / "train2017", overwrite=True, verbose=False, timeout=60.0, retries=3) for url in pool_tasks]
    if futs:
        with utils.ProgressBar(total=len(futs), desc="Downloading train2017") as pbar:
            for f in as_completed(futs):
                f.result()
                pbar.update(1)


In [ ]:
for im in val_images:
    p = RAW_DIR / "val2017" / im["file_name"]
    if files.is_dir(p):
        files.remove(p, missing_ok=True)

val_tasks = [
    (im.get("coco_url") or f"http://images.cocodataset.org/val2017/{im['file_name']}")
    for im in val_images
    if not files.is_file(RAW_DIR / "val2017" / im["file_name"])
]
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
    futs = [ex.submit(files.download, url, RAW_DIR / "val2017", overwrite=True, verbose=False, timeout=60.0, retries=3) for url in val_tasks]
    if futs:
        with utils.ProgressBar(total=len(futs), desc="Downloading val2017") as pbar:
            for f in as_completed(futs):
                f.result()
                pbar.update(1)


In [ ]:
POOL_YOLO_DIR = DATASET_DIR / "pool_yolo"
datasets.convert(target=POOL_YOLO_DIR, src_format="coco", dst_format="yolo", images=RAW_DIR / "train2017", annotations=POOL_JSON, include_crowd=False, overwrite=True, verbose=True)

In [ ]:
VAL_YOLO_DIR = DATASET_DIR / "val_yolo"
datasets.convert(target=VAL_YOLO_DIR, src_format="coco", dst_format="yolo", images=RAW_DIR / "val2017", annotations=VAL_JSON, include_crowd=False, overwrite=True, verbose=True)

In [ ]:
pool_dets = datasets.load(POOL_YOLO_DIR, format="yolo", verbose=False)
val_dets = datasets.load(VAL_YOLO_DIR, format="yolo", verbose=False)

utils.tabulate(
    reports.partition_summary({"Candidate Pool": POOL_YOLO_DIR, "Fixed Validation": VAL_YOLO_DIR}, format="yolo"),
    title="Bảng 4: Thống kê Pool & Validation sau chuyển đổi YOLO",
)

## 3. Khám phá dữ liệu, Khử trùng lặp & Trực quan hóa Ground Truth

In [ ]:
sample_pool_frames = [f for f in pool_dets if len(f) >= 3][:6]
sample_cells = graphics.compose([
    graphics.cell(frame, width=320, height=240, title=f"ID {frame.source_id} ({len(frame)} boxes)")
    for frame in sample_pool_frames
])
graphics.grid(sample_cells, rows=2, cols=3, title="Hình 1: Ảnh mẫu Ground Truth kèm Bounding Box (Candidate Pool)", save_path=REPORT_DIR / "fig_01_sample_ground_truth.png", dpi=300)

In [ ]:
dup_report = mining.find_duplicates(pool_dets, exact=True, method="phash", threshold=DEDUP_THRESHOLD)
clean_pool = mining.deduplicate(pool_dets, keep="best", threshold=DEDUP_THRESHOLD)

dedup_summary = {
    "pool_before": len(pool_dets),
    "duplicate_groups": len(dup_report.groups),
    "removed_duplicates": len(pool_dets) - len(clean_pool),
    "pool_after_dedup": len(clean_pool),
}
files.save(LOG_DIR / "03a_deduplication.json", dedup_summary, overwrite=True, verbose=False)
utils.tabulate(dedup_summary, orient="horizontal", title="Bảng 5a: Kiểm tra & Khử ảnh trùng lặp (MD5 + pHash)")

In [ ]:
pool_stats = mining.stats(clean_pool, quality=True, cooccurrence=False)
pool_summary = {
    "images": pool_stats.images,
    "total_boxes": pool_stats.boxes,
    "positive_images": pool_stats.images - pool_stats.empty_images,
    "background_images": pool_stats.empty_images,
    "classes_present": pool_stats.present_classes or len(pool_stats.boxes_per_class),
    "mean_boxes_per_image": round(pool_stats.boxes / max(1, pool_stats.images), 2),
    "imbalance_ratio": round(float(pool_stats.imbalance_ratio), 2),
}
files.save(LOG_DIR / "03b_ground_truth_analysis.json", pool_summary, overwrite=True, verbose=False)
utils.tabulate(pool_summary, orient="horizontal", title="Bảng 5b: Tổng quan Ground Truth sau khử trùng lặp")


In [ ]:
sorted_classes = sorted(pool_stats.boxes_per_class.items(), key=lambda kv: kv[1], reverse=True)
utils.tabulate(
    [{"rank": i + 1, "class": name, "boxes": cnt} for i, (name, cnt) in enumerate(sorted_classes[:15])],
    title="Bảng 6a: Top 15 lớp phổ biến nhất trong Candidate Pool",
)


In [ ]:
utils.tabulate(
    [{"rank": len(sorted_classes) - i, "class": name, "boxes": cnt} for i, (name, cnt) in enumerate(reversed(sorted_classes[-15:]))],
    title="Bảng 6b: Bottom 15 lớp hiếm nhất (Long-Tail) trong Candidate Pool",
)

In [ ]:
graphics.dataset_stats(
    POOL_YOLO_DIR,
    format="yolo",
    figsize=(14, 5),
    title="Hình 2: Phân bố số lượng đối tượng theo lớp trên Candidate Pool",
    save_path=REPORT_DIR / "fig_02_pool_class_distribution.png",
    dpi=300,
)

## 4. Khai phá dữ liệu — Chấm điểm, Phân cụm & 11 Chiến lược chọn mẫu (`klygo.mining`)

In [ ]:
density_scores = mining.score(clean_pool, weights={"boxes": 0.8, "coverage": 0.2})
weighted_scores = mining.score(clean_pool, weights={"quality": 0.3, "rarity": 0.4, "coverage": 0.2, "boxes": 0.1})

In [ ]:
cluster_div = mining.cluster(clean_pool, clusters=min(TRAIN_BUDGET, len(clean_pool)), seed=SEED)
cluster_grp = mining.cluster(clean_pool, clusters=N_CLUSTERS, seed=SEED)

In [ ]:
# 1. Random Sampling
sub_random = mining.select(clean_pool, method="random", limit=TRAIN_BUDGET, seed=SEED)
print(f"random: {len(sub_random)} images")

In [ ]:
# 2. Rare-First
sub_rare_first = mining.select(clean_pool, method="rare_first", limit=TRAIN_BUDGET)
print(f"rare_first: {len(sub_rare_first)} images")

In [ ]:
# 3. Density (ưu tiên mật độ box)
sub_density = mining.select(clean_pool, method="weighted", scores=density_scores, limit=TRAIN_BUDGET)
print(f"density: {len(sub_density)} images")

In [ ]:
# 4. Weighted Multi-Criteria (quality + rarity + coverage + boxes)
sub_weighted = mining.select(clean_pool, method="weighted", scores=weighted_scores, limit=TRAIN_BUDGET)
print(f"weighted: {len(sub_weighted)} images")

In [ ]:
# 5. Visual Quality
sub_quality = mining.select(clean_pool, method="quality", threshold=0.0, limit=TRAIN_BUDGET)
print(f"quality: {len(sub_quality)} images")

In [ ]:
# 6. Box Constraints (min_boxes + min_area_ratio)
sub_boxes = mining.select(clean_pool, method="boxes", min_boxes=2, min_area_ratio=0.002, limit=TRAIN_BUDGET)
print(f"boxes: {len(sub_boxes)} images")

In [ ]:
# 7. Class Coverage (tối đa hóa độ phủ lớp, tự động bù đủ TRAIN_BUDGET)
sub_coverage = mining.select(clean_pool, method="coverage", limit=TRAIN_BUDGET)
print(f"coverage: {len(sub_coverage)} images")

In [ ]:
# 8. Per-Class Quota
sub_quota = mining.select(clean_pool, method="quota", per_class=10, limit=TRAIN_BUDGET)
print(f"quota: {len(sub_quota)} images")

In [ ]:
# 9. Diversity K-Means++ (550 cụm x 1 đại diện)
sub_diversity = mining.select(clean_pool, method="diversity", cluster_result=cluster_div, per_cluster=1, limit=TRAIN_BUDGET, seed=SEED)
print(f"diversity: {len(sub_diversity)} images")

In [ ]:
# 10. Clustering K-Means++ (25 cụm x quota, có cơ chế tự bù)
per_cluster_quota = max(1, (TRAIN_BUDGET + N_CLUSTERS - 1) // N_CLUSTERS)
sub_clustering = mining.select(clean_pool, method="diversity", cluster_result=cluster_grp, per_cluster=per_cluster_quota, limit=TRAIN_BUDGET, seed=SEED)
print(f"clustering: {len(sub_clustering)} images")

In [ ]:
# 11. Class Balancing (mining.balance)
balanced_pool = mining.balance(clean_pool, target="median", count_by="images", seed=SEED)
sub_balance = mining.select(balanced_pool, method="random", limit=TRAIN_BUDGET, seed=SEED)
print(f"balance: {len(sub_balance)} images (from {len(balanced_pool)} balanced pool)")

## 5. Thống kê, Trực quan hóa & Xuất 11 bộ dữ liệu YOLO

In [ ]:
selections = {
    "random": sub_random,
    "rare_first": sub_rare_first,
    "density": sub_density,
    "weighted": sub_weighted,
    "quality": sub_quality,
    "boxes": sub_boxes,
    "coverage": sub_coverage,
    "quota": sub_quota,
    "diversity": sub_diversity,
    "clustering": sub_clustering,
    "balance": sub_balance,
}
files.save(LOG_DIR / "04_mining_selections.json", {k: [f.source_id for f in v] for k, v in selections.items()}, overwrite=True, verbose=False)

In [ ]:
mining_rows = reports.mining_breakdown(selections)
for row in mining_rows:
    delta = mining.compare(clean_pool, selections[row["strategy"]], quality=False)
    row["class_coverage"] = delta.after.present_classes or len(delta.after.boxes_per_class)
    row["imbalance_ratio"] = round(float(delta.after.imbalance_ratio), 2)

files.save(REPORT_DIR / "table_mining_breakdown.csv", mining_rows, overwrite=True, verbose=False)
utils.tabulate(mining_rows, title="Bảng 7: Thống kê đặc trưng tập dữ liệu chọn bởi 11 chiến lược Data Mining")


In [ ]:
rep_cells = graphics.compose([
    graphics.cell(
        next((f for f in subset if len(f) > 0), subset[0]),
        width=280,
        height=210,
        title=f"{name} ({len(next((f for f in subset if len(f) > 0), subset[0]))} boxes)",
    )
    for name, subset in selections.items()
])
graphics.grid(rep_cells, rows=3, cols=4, title="Hình 3: Ảnh đại diện kèm Bounding Box của 11 chiến lược Data Mining", save_path=REPORT_DIR / "fig_03_mining_representative_samples.png", dpi=300)

In [ ]:
dataset_yamls = {}
for strategy, subset in selections.items():
    strat_dir = DATASET_DIR / strategy
    subset.export(str(strat_dir), format="yolo", image=True, overwrite=True)

    yaml_path = strat_dir / "data.yaml"
    data_cfg = config.load(yaml_path, verbose=False)
    data_cfg = config.set(data_cfg, "path", str(strat_dir))
    data_cfg = config.set(data_cfg, "train", "images")
    data_cfg = config.set(data_cfg, "val", str(VAL_YOLO_DIR / "images"))
    files.save(yaml_path, data_cfg.to_dict(), overwrite=True, verbose=False)
    dataset_yamls[strategy] = str(yaml_path)

In [ ]:
partition_rows = reports.partition_summary({"val2017": VAL_YOLO_DIR, **{k: DATASET_DIR / k for k in selections}}, format="yolo")
files.save(LOG_DIR / "05_partition_summary.json", partition_rows, overwrite=True, verbose=False)
utils.tabulate(partition_rows, title="Bảng 8: Kiểm tra toàn vẹn & số lượng ảnh/box 11 bộ YOLO đã xuất")

In [ ]:
fig_part = graphics.partitions(
    partition_rows,
    figsize=(12, 4.5),
    title="Hình 4a: Phân bố ảnh giữa Validation & 11 tập Train",
    save_path=REPORT_DIR / "fig_04a_partitions.png",
    show=False,
    return_figure=True,
)

In [ ]:
fig_mining = graphics.mining(
    mining_rows,
    kind="counts",
    figsize=(12, 4.5),
    title="Hình 4b: Ảnh dương tính / Ảnh nền & Tổng số box theo 11 chiến lược",
    save_path=REPORT_DIR / "fig_04b_mining_counts.png",
    show=False,
    return_figure=True,
)

In [ ]:
overview_cells = graphics.compose([
    graphics.cell(fig_part, title="(a) Dataset Partitions"),
    graphics.cell(fig_mining, title="(b) Mining Strategies"),
])
graphics.grid(overview_cells, rows=2, cols=1, title="Hình 4: Tổng quan phân bố dữ liệu & 11 chiến lược Data Mining", save_path=REPORT_DIR / "fig_04_dataset_and_mining_overview.png", dpi=300)

## 6. Huấn luyện 11 Student YOLO11n & Đánh giá trên `val2017`

In [ ]:
run_dirs, best_checkpoints = {}, {}

if RUN_TRAINING:
    for strategy in selections:
        print(f"\n{'=' * 20} TRAIN: {strategy} {'=' * 20}")
        detector = models.load(MODEL_NAME)
        run_name = f"yolo11n_{strategy}"
        detector.model.train(
            data=dataset_yamls[strategy],
            project=str(RUNS_DIR),
            name=run_name,
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            workers=WORKERS,
            device=DEVICE,
            seed=SEED,
            deterministic=True,
            patience=EPOCHS,
            exist_ok=True,
        )
        run_dir = RUNS_DIR / run_name
        run_dirs[strategy] = [str(run_dir)]
        ckpt_src = run_dir / "weights" / "best.pt" if files.is_file(run_dir / "weights" / "best.pt") else run_dir / "weights" / "last.pt"
        ckpt_dst = CHECKPOINT_DIR / f"{strategy}_best.pt"
        files.copy(ckpt_src, ckpt_dst, overwrite=True)
        best_checkpoints[strategy] = str(ckpt_dst)
        del detector
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
if run_dirs:
    c = reports.load(next(iter(run_dirs.values()))[0], format="yolo")["config"]
    hyper_table = {
        "Mô hình": c.get("model"),
        "Kích thước ảnh / Batch / Epoch": f"imgsz={c.get('imgsz')}; batch={c.get('batch')}; epochs={c.get('epochs')} (patience={c.get('patience')})",
        "Optimizer": f"{c.get('optimizer')} (lr0={c.get('lr0')}, momentum={c.get('momentum')})",
        "Weight decay & Warmup": f"weight_decay={c.get('weight_decay')}; warmup_epochs={c.get('warmup_epochs')}",
        "Trọng số Loss": f"box={c.get('box')}, cls={c.get('cls')}, dfl={c.get('dfl')}",
        "Augmentation": f"mosaic={c.get('mosaic')} (close_mosaic={c.get('close_mosaic')}), fliplr={c.get('fliplr')}, scale={c.get('scale')}, AMP={c.get('amp')}",
        "Kiểm soát ngẫu nhiên": f"deterministic={c.get('deterministic')}, seed={c.get('seed')}",
    }
    utils.tabulate(hyper_table, title="Bảng 9: Cấu hình siêu tham số huấn luyện từ args.yaml", key_header="Tham số", value_header="Giá trị")

In [ ]:
training_records = []
val_results = {}

for strategy, ckpt_path in best_checkpoints.items():
    detector = models.load(ckpt_path)
    yolo_val = detector.model.val(
        data=dataset_yamls[strategy],
        split="val",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        verbose=False,
    )

    mp = float(yolo_val.box.mp)
    mr = float(yolo_val.box.mr)
    f1 = float(metrics.f1_score(mp, mr))
    map50 = float(yolo_val.box.map50)
    map75 = float(yolo_val.box.map75)
    map50_95 = float(yolo_val.box.map)

    record = {
        "strategy": strategy,
        "train_images": len(selections[strategy]),
        "precision": round(mp, 4),
        "recall": round(mr, 4),
        "f1": round(f1, 4),
        "mAP50": round(map50, 4),
        "mAP75": round(map75, 4),
        "mAP50-95": round(map50_95, 4),
    }
    training_records.append(record)
    val_results[strategy] = yolo_val
    files.save(LOG_DIR / f"eval_{strategy}.json", record, overwrite=True, verbose=False)
    del detector
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 7. Bảng tổng hợp kết quả & Phân tích chi tiết (`klygo.reports` + `utils.tabulate`)

In [ ]:
if run_dirs:
    agg = reports.aggregate_runs(run_dirs, format="yolo", best="auto")
    files.save(REPORT_DIR / "table_training_logs.csv", agg["table"], overwrite=True, verbose=False)
    utils.tabulate(agg["table"], title="Bảng 10: Tổng hợp nhật ký huấn luyện từ results.csv")

In [ ]:
leaderboard_rows = []
if training_records:
    base_map = next((r["mAP50-95"] for r in training_records if r["strategy"] == "random"), 0.0)
    for rank, row in enumerate(sorted(training_records, key=lambda r: r["mAP50-95"], reverse=True), start=1):
        delta = round(row["mAP50-95"] - base_map, 4)
        delta_pct = round((100.0 * delta / base_map) if base_map else 0.0, 2)
        leaderboard_rows.append({"rank": rank, **row, "delta_vs_random": delta, "delta_pct": delta_pct})
    files.save(REPORT_DIR / "table_leaderboard.csv", leaderboard_rows, overwrite=True, verbose=False)
    utils.tabulate(leaderboard_rows, title="Bảng 11: Bảng xếp hạng 11 chiến lược Data Mining trên COCO val2017")

In [ ]:
if leaderboard_rows:
    best_strategy = leaderboard_rows[0]["strategy"]
    best_val = val_results[best_strategy]
    if hasattr(best_val.box, "all_ap") and best_val.box.all_ap is not None and len(best_val.box.all_ap):
        ious = np.linspace(0.5, 0.95, 10)
        iou_maps = best_val.box.all_ap.mean(axis=0)
        map50 = float(iou_maps[0]) if len(iou_maps) else float(best_val.box.map50)
        iou_rows = []
        for iou, m in zip(ious, iou_maps):
            iou_f = round(float(iou), 2)
            m_f = round(float(m), 4)
            delta = round(m_f - map50, 4)
            retention = f"{round(100.0 * m_f / max(1e-6, map50), 1)}%"
            if iou_f <= 0.60:
                tier = "Cơ bản (Loose)"
            elif iou_f <= 0.75:
                tier = "Tiêu chuẩn (COCO)"
            else:
                tier = "Khắt khe (Strict)"
            iou_rows.append({
                "IoU": f"{iou_f:.2f}",
                "mAP": m_f,
                "delta_vs_50": delta,
                "retention": retention,
                "tier": tier,
            })
    else:
        map50 = float(best_val.box.map50)
        map75 = float(best_val.box.map75)
        map_all = float(best_val.box.map)
        iou_rows = [
            {"IoU": "0.50", "mAP": round(map50, 4), "delta_vs_50": 0.0, "retention": "100.0%", "tier": "Cơ bản (Loose)"},
            {"IoU": "0.75", "mAP": round(map75, 4), "delta_vs_50": round(map75 - map50, 4), "retention": f"{round(100.0 * map75 / max(1e-6, map50), 1)}%", "tier": "Tiêu chuẩn (COCO)"},
            {"IoU": "0.50:0.95", "mAP": round(map_all, 4), "delta_vs_50": round(map_all - map50, 4), "retention": f"{round(100.0 * map_all / max(1e-6, map50), 1)}%", "tier": "Tổng hợp (COCO)"},
        ]
    files.save(REPORT_DIR / "table_iou_breakdown.csv", iou_rows, overwrite=True, verbose=False)
    utils.tabulate(iou_rows, title=f"Bảng 12a: Phân rã mAP theo 10 ngưỡng IoU — {best_strategy}")

In [ ]:
per_class = []
if leaderboard_rows:
    best_strategy = leaderboard_rows[0]["strategy"]
    best_val = val_results[best_strategy]
    mean_ap = float(best_val.box.map) if hasattr(best_val.box, "map") else 0.0
    has_all_ap = hasattr(best_val.box, "all_ap") and best_val.box.all_ap is not None and len(best_val.box.all_ap)

    if hasattr(best_val.box, "ap_class_index") and best_val.box.ap_class_index is not None:
        for idx, cls_id in enumerate(best_val.box.ap_class_index):
            cls_name = best_val.names.get(cls_id, str(cls_id)) if hasattr(best_val, "names") else str(cls_id)
            ap_val = float(best_val.box.maps[idx]) if idx < len(best_val.box.maps) else 0.0
            ap50_val = float(best_val.box.all_ap[idx, 0]) if has_all_ap and idx < len(best_val.box.all_ap) else None
            per_class.append({
                "class": cls_name,
                "class_id": int(cls_id),
                "AP50": round(ap50_val, 4) if ap50_val is not None else "-",
                "AP50-95": round(ap_val, 4),
                "delta_mean": round(ap_val - mean_ap, 4),
            })
        per_class = sorted(per_class, key=lambda x: x["AP50-95"], reverse=True)

    if per_class:
        top10_rows = []
        for rk, c in enumerate(per_class[:10], start=1):
            grade = "Xuất sắc" if c["AP50-95"] >= 0.60 else ("Tốt" if c["AP50-95"] >= 0.40 else "Đạt")
            top10_rows.append({
                "rank": f"#{rk}",
                "class_id": c["class_id"],
                "class": c["class"],
                "AP50": c["AP50"],
                "AP50-95": c["AP50-95"],
                "delta_mean": c["delta_mean"],
                "grade": grade,
            })
        files.save(REPORT_DIR / "table_top10_classes.csv", top10_rows, overwrite=True, verbose=False)
        utils.tabulate(
            top10_rows,
            title=f"Bảng 12b: Top 10 lớp có AP cao nhất — {best_strategy}",
        )
    else:
        print("Không có lớp nào có đủ mẫu validation để xếp hạng Top 10.")

In [ ]:
if leaderboard_rows and per_class:
    total_cls = len(per_class)
    bottom10 = per_class[-10:]
    bottom10_rows = []
    for offset, c in enumerate(reversed(bottom10), start=1):
        tier = "Cực khó (AP=0)" if c["AP50-95"] == 0.0 else ("Thách thức cao" if c["AP50-95"] < 0.15 else "Cần thêm dữ liệu")
        bottom10_rows.append({
            "rank": f"#{total_cls - offset + 1}",
            "class_id": c["class_id"],
            "class": c["class"],
            "AP50": c["AP50"],
            "AP50-95": c["AP50-95"],
            "deficit_mean": c["delta_mean"],
            "difficulty": tier,
        })
    files.save(REPORT_DIR / "table_bottom10_classes.csv", bottom10_rows, overwrite=True, verbose=False)
    utils.tabulate(
        bottom10_rows,
        title=f"Bảng 12c: Bottom 10 lớp khó nhất (Cần khai phá thêm) — {best_strategy}",
    )

## 8. Biểu đồ hội tụ, So sánh hiệu năng & Phân tích định tính (`klygo.graphics`)

In [ ]:
if run_dirs:
    fig_box = graphics.curves(agg, metric="box_loss", split="train", band=False, show=False, return_figure=True)
    fig_cls = graphics.curves(agg, metric="cls_loss", split="train", band=False, show=False, return_figure=True)
    fig_dfl = graphics.curves(agg, metric="dfl_loss", split="train", band=False, show=False, return_figure=True)
    loss_cells = graphics.compose([
        graphics.cell(fig_box, title="(a) Train Box Loss"),
        graphics.cell(fig_cls, title="(b) Train Cls Loss"),
        graphics.cell(fig_dfl, title="(c) Train DFL Loss"),
    ])
    graphics.grid(loss_cells, rows=1, cols=3, title="Hình 5: Quá trình hội tụ Training Loss của 11 chiến lược", save_path=REPORT_DIR / "fig_05_training_loss_curves.png", dpi=300)

In [ ]:
if run_dirs:
    fig_m50 = graphics.curves(agg, metric="mAP50", band=False, show=False, return_figure=True)
    fig_m95 = graphics.curves(agg, metric="mAP50-95", band=False, show=False, return_figure=True)
    map_cells = graphics.compose([
        graphics.cell(fig_m50, title="(a) Validation mAP@50"),
        graphics.cell(fig_m95, title="(b) Validation mAP@50-95"),
    ])
    graphics.grid(map_cells, rows=1, cols=2, title="Hình 6: Quá trình tăng trưởng Validation mAP qua các Epoch", save_path=REPORT_DIR / "fig_06_validation_map_curves.png", dpi=300)

In [ ]:
if leaderboard_rows:
    graphics.bars(
        leaderboard_rows,
        metrics=("mAP50-95",),
        error_bars=False,
        figsize=(12, 5),
        title="Hình 7: So sánh mAP@50-95 giữa 11 chiến lược Data Mining",
        save_path=REPORT_DIR / "fig_07_map50_95_comparison.png",
        dpi=300,
    )

In [ ]:
if leaderboard_rows:
    graphics.bars(
        leaderboard_rows,
        metrics=("precision", "recall", "f1", "mAP50", "mAP50-95"),
        error_bars=False,
        show_values=False,
        figsize=(14, 5.2),
        title="Hình 8: So sánh đa chỉ số (P / R / F1 / mAP@50 / mAP@50-95)",
        save_path=REPORT_DIR / "fig_08_multi_metric_comparison.png",
        dpi=300,
    )
    multi_metric_rows = [
        {
            "rank": r["rank"],
            "strategy": r["strategy"],
            "P": round(float(r["precision"]), 4),
            "R": round(float(r["recall"]), 4),
            "F1": round(float(r["f1"]), 4),
            "mAP50": round(float(r["mAP50"]), 4),
            "mAP50-95": round(float(r["mAP50-95"]), 4),
        }
        for r in leaderboard_rows
    ]
    files.save(REPORT_DIR / "table_multi_metric_comparison.csv", multi_metric_rows, overwrite=True, verbose=False)
    utils.tabulate(
        multi_metric_rows,
        title="Bảng 11b: So sánh chi tiết đa chỉ số hiệu năng — 11 chiến lược Data Mining",
    )

In [ ]:
if leaderboard_rows:
    best_det = models.load(best_checkpoints[best_strategy])
    sample_dets = val_dets[:min(3, len(val_dets))].copy()
    sample_preds = sample_dets.copy()
    sample_preds.repredict(best_det, threshold=0.35, verbose=False)

    for idx in range(len(sample_dets)):
        preview = graphics.compare(
            sample_dets[idx],
            sample_preds[idx],
            titles=("Ground Truth", f"Prediction ({best_strategy})"),
            mode="save",
            output_path=REPORT_DIR / f"fig_09_qualitative_sample_{idx + 1}.png",
            overwrite=True,
        )
        graphics.show(preview, title=f"Hình 9.{idx + 1}: Ground Truth vs {best_strategy}")
    del best_det
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 9. Kiểm chứng độ ổn định 3 Seeds (`random` + Top-3 chiến lược tốt nhất)

In [ ]:
multi_seed_runs = {}

if RUN_TRAINING and RUN_MULTI_SEED and leaderboard_rows:
    top_strategies = []
    for s in [r["strategy"] for r in leaderboard_rows[:3]] + ["random"]:
        if s not in top_strategies:
            top_strategies.append(s)

    for strategy in top_strategies:
        multi_seed_runs[strategy] = list(run_dirs[strategy])  # tái sử dụng seed 42 đã chạy
        for s_val in MULTI_SEEDS:
            if s_val == SEED:
                continue
            print(f"\n{'=' * 18} MULTI-SEED: {strategy} (seed={s_val}) {'=' * 18}")
            detector = models.load(MODEL_NAME)
            run_name = f"yolo11n_{strategy}_seed{s_val}"
            detector.model.train(
                data=dataset_yamls[strategy],
                project=str(RUNS_DIR),
                name=run_name,
                epochs=EPOCHS,
                imgsz=IMGSZ,
                batch=BATCH,
                workers=WORKERS,
                device=DEVICE,
                seed=s_val,
                deterministic=True,
                patience=EPOCHS,
                exist_ok=True,
            )
            run_dir = RUNS_DIR / run_name
            multi_seed_runs[strategy].append(str(run_dir))
            ckpt_src = run_dir / "weights" / "best.pt" if files.is_file(run_dir / "weights" / "best.pt") else run_dir / "weights" / "last.pt"
            files.copy(ckpt_src, CHECKPOINT_DIR / f"{strategy}_seed{s_val}_best.pt", overwrite=True)
            del detector
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

In [ ]:
multi_agg = None
if multi_seed_runs:
    pair = (best_strategy, "random") if best_strategy != "random" else None
    multi_agg = reports.aggregate_runs(multi_seed_runs, format="yolo", best="auto", compare_pair=pair)
    files.save(REPORT_DIR / "table_multi_seed_summary.csv", multi_agg["table"], overwrite=True, verbose=False)
    files.save(LOG_DIR / "07a_multi_seed_delta.json", multi_agg["delta"], overwrite=True, verbose=False)
    utils.tabulate(multi_agg["table"], title="Bảng 14a: Kiểm chứng ổn định 3 Seeds (Mean ± Std — random + Top-3)")

In [ ]:
if multi_agg:
    files.save(REPORT_DIR / "table_multi_seed_per_seed.csv", multi_agg["per_seed"], overwrite=True, verbose=False)
    utils.tabulate(multi_agg["per_seed"], title="Bảng 14b: Kết quả chi tiết từng Seed (Phụ lục)")

In [ ]:
if multi_agg:
    fig_ms_loss = graphics.curves(multi_agg, metric="box_loss", split="train", band=True, show=False, return_figure=True)
    fig_ms_map = graphics.curves(multi_agg, metric="mAP50-95", band=True, show=False, return_figure=True)
    ms_cells = graphics.compose([
        graphics.cell(fig_ms_loss, title="(a) Train Box Loss (Mean ± Std)"),
        graphics.cell(fig_ms_map, title="(b) Validation mAP@50-95 (Mean ± Std)"),
    ])
    graphics.grid(ms_cells, rows=1, cols=2, title="Hình 10: Độ ổn định qua 3 Seeds (random + Top-3)", save_path=REPORT_DIR / "fig_10_multi_seed_curves.png", dpi=300)

In [ ]:
if multi_agg:
    graphics.bars(
        multi_agg["table"],
        metrics=("mAP50-95",),
        error_bars=True,
        title="Hình 11: So sánh Validation mAP@50-95 (Mean ± Std qua 3 Seeds)",
        save_path=REPORT_DIR / "fig_11_multi_seed_bars.png",
        dpi=300,
    )

## 10. Khảo sát hiệu quả dữ liệu theo ngân sách (`BUDGET_GRID`: 250 / 550 / 1000 ảnh)

In [ ]:
scaling_rows = []

if RUN_TRAINING and RUN_BUDGET_SCALING and leaderboard_rows:
    scale_strategies = ["random"] if best_strategy == "random" else ["random", best_strategy]
    for budget_k in BUDGET_GRID:
        for strategy in scale_strategies:
            if budget_k == TRAIN_BUDGET:
                base_rec = next(r for r in training_records if r["strategy"] == strategy)
                scaling_rows.append({"label": f"{strategy}@{budget_k}", "budget": budget_k, **base_rec})
                continue

            if strategy == "random":
                sub_k = mining.select(clean_pool, method="random", limit=budget_k, seed=SEED)
            elif strategy == "rare_first":
                sub_k = mining.select(clean_pool, method="rare_first", limit=budget_k)
            elif strategy == "density":
                sub_k = mining.select(clean_pool, method="weighted", scores=density_scores, limit=budget_k)
            elif strategy == "weighted":
                sub_k = mining.select(clean_pool, method="weighted", scores=weighted_scores, limit=budget_k)
            elif strategy == "quality":
                sub_k = mining.select(clean_pool, method="quality", threshold=0.0, limit=budget_k)
            elif strategy == "boxes":
                sub_k = mining.select(clean_pool, method="boxes", min_boxes=2, min_area_ratio=0.002, limit=budget_k)
            elif strategy == "coverage":
                sub_k = mining.select(clean_pool, method="coverage", limit=budget_k)
            elif strategy == "quota":
                sub_k = mining.select(clean_pool, method="quota", per_class=max(1, budget_k // 55), limit=budget_k)
            elif strategy == "diversity":
                cl_k = mining.cluster(clean_pool, clusters=min(budget_k, len(clean_pool)), seed=SEED)
                sub_k = mining.select(clean_pool, method="diversity", cluster_result=cl_k, per_cluster=1, limit=budget_k, seed=SEED)
            elif strategy == "clustering":
                q_k = max(1, (budget_k + N_CLUSTERS - 1) // N_CLUSTERS)
                sub_k = mining.select(clean_pool, method="diversity", cluster_result=cluster_grp, per_cluster=q_k, limit=budget_k, seed=SEED)
            else:
                sub_k = mining.select(balanced_pool, method="random", limit=budget_k, seed=SEED)

            strat_dir = DATASET_DIR / f"scale_{strategy}_{budget_k}"
            sub_k.export(str(strat_dir), format="yolo", image=True, overwrite=True)
            yaml_path = strat_dir / "data.yaml"
            data_cfg = config.load(yaml_path, verbose=False)
            data_cfg = config.set(data_cfg, "path", str(strat_dir))
            data_cfg = config.set(data_cfg, "train", "images")
            data_cfg = config.set(data_cfg, "val", str(VAL_YOLO_DIR / "images"))
            files.save(yaml_path, data_cfg.to_dict(), overwrite=True, verbose=False)

            detector = models.load(MODEL_NAME)
            run_name = f"yolo11n_scale_{strategy}_{budget_k}"
            detector.model.train(
                data=str(yaml_path),
                project=str(RUNS_DIR),
                name=run_name,
                epochs=EPOCHS,
                imgsz=IMGSZ,
                batch=BATCH,
                workers=WORKERS,
                device=DEVICE,
                seed=SEED,
                deterministic=True,
                patience=EPOCHS,
                exist_ok=True,
            )
            run_dir = RUNS_DIR / run_name
            ckpt_src = run_dir / "weights" / "best.pt" if files.is_file(run_dir / "weights" / "best.pt") else run_dir / "weights" / "last.pt"
            ckpt_dst = CHECKPOINT_DIR / f"scale_{strategy}_{budget_k}_best.pt"
            files.copy(ckpt_src, ckpt_dst, overwrite=True)

            eval_det = models.load(ckpt_dst)
            y_val = eval_det.model.val(
                data=str(yaml_path),
                split="val",
                imgsz=IMGSZ,
                batch=BATCH,
                device=DEVICE,
                verbose=False,
            )
            mp = float(y_val.box.mp)
            mr = float(y_val.box.mr)
            f1 = float(metrics.f1_score(mp, mr))

            scaling_rows.append({
                "label": f"{strategy}@{budget_k}",
                "budget": budget_k,
                "strategy": strategy,
                "train_images": len(sub_k),
                "precision": round(mp, 4),
                "recall": round(mr, 4),
                "f1": round(f1, 4),
                "mAP50": round(float(y_val.box.map50), 4),
                "mAP75": round(float(y_val.box.map75), 4),
                "mAP50-95": round(float(y_val.box.map), 4),
            })
            del detector, eval_det
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

In [ ]:
if scaling_rows:
    files.save(REPORT_DIR / "table_budget_scaling.csv", scaling_rows, overwrite=True, verbose=False)
    utils.tabulate(scaling_rows, title="Bảng 15: Hiệu quả sử dụng dữ liệu theo ngân sách (Budget Scaling)")

In [ ]:
if scaling_rows:
    bar_rows = [{"strategy": r["label"], "mAP50": r["mAP50"], "mAP50-95": r["mAP50-95"]} for r in scaling_rows]
    graphics.bars(
        bar_rows,
        metrics=("mAP50", "mAP50-95"),
        error_bars=False,
        show_values=False,
        figsize=(11, 4.8),
        title="Hình 12: So sánh hiệu năng theo ngân sách dữ liệu (250 / 550 / 1000 ảnh)",
        save_path=REPORT_DIR / "fig_12_budget_scaling.png",
        dpi=300,
    )

## 11. Kiểm toán tổng hợp & Đóng gói ZIP bàn giao (`klygo.files` + `klygo.archive`)

In [ ]:
mining_by_strat = {r["strategy"]: r for r in mining_rows}
lb_by_strat = {r["strategy"]: r for r in leaderboard_rows} if leaderboard_rows else {}

final_audit_rows = [
    {
        "strategy": s,
        "images": mining_by_strat[s]["images"],
        "boxes": mining_by_strat[s]["boxes"],
        "class_coverage": mining_by_strat[s]["class_coverage"],
        "imbalance_ratio": mining_by_strat[s]["imbalance_ratio"],
        **({
            "rank": lb_by_strat[s]["rank"],
            "precision": lb_by_strat[s]["precision"],
            "recall": lb_by_strat[s]["recall"],
            "f1": lb_by_strat[s]["f1"],
            "mAP50": lb_by_strat[s]["mAP50"],
            "mAP50-95": lb_by_strat[s]["mAP50-95"],
            "delta_pct": lb_by_strat[s]["delta_pct"],
        } if s in lb_by_strat else {}),
    }
    for s in selections
]
files.save(REPORT_DIR / "table_final_audit.csv", final_audit_rows, overwrite=True, verbose=False)
utils.tabulate(final_audit_rows, title="Bảng 13: Bảng tổng hợp kiểm toán toàn bộ 11 chiến lược")

In [ ]:
files.save(ROOT / "README.md", f"# COCO Ground-Truth Data Mining + YOLO11n (Klygo)\n- Pool: {POOL_LIMIT} | Val: {VAL_LIMIT} | Budget: {TRAIN_BUDGET} | Strategies: 11 | Model: {MODEL_NAME} | Epochs: {EPOCHS}\n", overwrite=True, verbose=False)
files.save(ROOT / "requirements.txt", "klygo[ultralytics]\nnumpy\ntorch\n", overwrite=True, verbose=False)

ZIP_PATH = files.path("/content/COCO_GroundTruth_DataMining_YOLO11n_Klygo.zip")
PACKAGE_ROOT = ROOT / "package"
files.remove(ZIP_PATH, missing_ok=True)
files.remove(PACKAGE_ROOT, missing_ok=True)
files.mkdir(PACKAGE_ROOT)

# Chỉ đóng gói logs, reports, checkpoints và runs (kèm weights); KHÔNG đóng gói dataset ảnh để ZIP gọn nhẹ
for sub in ["logs", "reports", "checkpoints", "runs"]:
    src = ARTIFACTS / sub
    if files.is_dir(src):
        files.copy(src, PACKAGE_ROOT / "artifacts" / sub, overwrite=True)
files.copy(ROOT / "README.md", PACKAGE_ROOT / "README.md", overwrite=True)
files.copy(ROOT / "requirements.txt", PACKAGE_ROOT / "requirements.txt", overwrite=True)

In [ ]:
inventory = [
    {"path": str(p.relative_to(PACKAGE_ROOT)), "size": files.size(p), "sha256": files.hash(p, algorithm="sha256")}
    for p in files.find(PACKAGE_ROOT, recursive=True)
]
files.save(PACKAGE_ROOT / "artifacts" / "logs" / "08_package_inventory.json", inventory, overwrite=True, verbose=False)
archive.compress(PACKAGE_ROOT, ZIP_PATH, overwrite=True, verbose=True)


In [ ]:
zip_info = {
    "zip_path": str(ZIP_PATH),
    "size": utils.human_size(files.size(ZIP_PATH)),
    "entries": len(archive.list_files(ZIP_PATH)),
    "sha256": files.hash(ZIP_PATH, algorithm="sha256"),
}
utils.tabulate(zip_info, orient="horizontal", title="Bảng 16: Thông tin gói bàn giao ZIP")

try:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP_PATH))
except ImportError:
    print("ZIP saved locally at:", ZIP_PATH)


## 12. Tự động ngắt runtime Colab L4 sau khi chạy xong

In [ ]:
# Chờ 15 giây để trình duyệt hoàn tất tải ZIP rồi tự động ngắt GPU runtime tránh tốn Compute Units
import time

AUTO_SHUTDOWN = True
if AUTO_SHUTDOWN:
    try:
        from google.colab import runtime
        print("All experiments finished! Waiting 15s before shutting down Colab runtime...")
        time.sleep(15)
        runtime.unassign()
    except ImportError:
        print("Not running in Google Colab — skipping runtime.unassign().")